# D.R.O.N.A. — 11 Sim-to-Real Handoff

Demonstrates the embodied layer and the clean sim→real seam (Objective 3:
robot-independent framework). Covers:

1. Gesture policy rollout + smoothness in `StubEnv`/MuJoCo (works on the laptop)
2. The ROS2 action interface the policy node exposes (`ExecuteGesture.action`)
3. The Gazebo Harmonic / Isaac Sim handoff (see `docs/sim_setup_*.md`)

The Python rollout runs anywhere; ROS2/sim cells are documented commands to run
on Ubuntu 22.04 + ROS2 Humble.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
from drona.utils.logging import setup_logging; setup_logging('WARNING')

# Roll out every gesture with the keyframe policy and measure quality.
from drona.interaction.sim_eval import evaluate_keyframe_baseline
report = evaluate_keyframe_baseline()
print(f'Success rate: {report.success_rate:.2f}')
print(f'Mean jerk:    {report.mean_jerk:.4f} rad/s^3')
print(f'Mean path:    {report.mean_path_length:.3f} rad')
for g, m in report.per_gesture.items():
    print(f'  {g:<10} success={m.success} jerk={m.jerk:.3f} apex_err={m.apex_error:.3f}')

In [ ]:
# Compare a trained ACT checkpoint to the keyframe baseline, if one exists.
from pathlib import Path
from drona.utils.settings import settings
ckpt = settings.data_dir / 'checkpoints'
try:
    from drona.interaction.sim_eval import evaluate_policy, compare_policies
    from drona.interaction.act_policy import PolicyRouter, KeyframePolicy
    if ckpt.exists() and any(ckpt.iterdir()):
        router = PolicyRouter(checkpoint_base_dir=str(ckpt), device='cpu')
        delta = compare_policies(
            lambda g: KeyframePolicy(g),
            lambda g: router.get_policy(g),
        )
        print('ACT vs keyframe deltas:', delta)
    else:
        print('No ACT checkpoint found at', ckpt)
        print('Train one in notebook 07 (Colab T4) and copy it here to compare.')
except Exception as exc:
    print('Comparison note:', exc)

## ROS2 + simulation handoff (Ubuntu 22.04 + ROS2 Humble)

The policy node exposes a clean action interface that swaps to a real robot
driver unchanged — the only real-world step left to the student.

```bash
# Build
cd ros2_ws && colcon build --symlink-install && source install/setup.bash

# Gazebo Harmonic (low-VRAM, recommended on the GTX-1650)
ros2 launch drona_bringup drona_gazebo.launch.py

# Full system + RViz + record an end-to-end interaction
ros2 launch drona_bringup drona_system.launch.py use_rviz:=true record:=true bag_path:=demo_run

# Drive a gesture through the action server (streams feedback)
ros2 action send_goal /drona/execute_gesture_action \
  drona_msgs/action/ExecuteGesture "{gesture_label: 'greet'}" --feedback

# Replay the recorded session
ros2 bag play demo_run
```

Isaac Sim (≥8 GB VRAM) and the cloud-GPU recipe are in `docs/sim_setup_isaac.md`;
the full interface contract is in `docs/ros2_topics_actions.md`.

**Sim-to-real:** the same `/drona/joint_states` stream drives `StubEnv` here,
the URDF in Gazebo/Isaac, and — in Phase 2 — the SO-100 arm driver. No advising
or policy logic changes; only the `arm_interface` backend is swapped.